# 🔧 Gravitational Lensing - FIXED v2 (Auto-Compute Statistics)

## ⚠️ CRITICAL FIX:
This version **automatically computes the correct global mean/std** from your actual dataset!

**Previous issue**: Hardcoded statistics were wrong for your data.

---

In [ ]:
%%capture
!pip install -q scikit-learn seaborn tqdm

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json
from datetime import datetime
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
from google.colab import drive

print("✅ Imports successful")

## Mount Drive & Extract Dataset

In [ ]:
drive.mount('/content/drive')

In [ ]:
import zipfile

DRIVE_DATASET_PATH = '/content/drive/MyDrive/data/dataset.zip'
LOCAL_EXTRACT_PATH = '/content/dataset'

if not os.path.exists(DRIVE_DATASET_PATH):
    print(f"❌ ERROR: Dataset not found at {DRIVE_DATASET_PATH}")
else:
    print(f"✅ Dataset found: {DRIVE_DATASET_PATH}")
    print("\n📦 Extracting...")
    with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("✅ Extraction complete!")

## 🔥 CRITICAL: Compute Global Statistics from YOUR Dataset

This is the **KEY FIX** - we compute mean/std from the actual data!

In [ ]:
print("="*70)
print("COMPUTING GLOBAL STATISTICS (This is the KEY FIX!)")
print("="*70)

DATA_DIR = Path('/content/dataset/train')
CLASS_NAMES = ['no', 'sphere', 'vort']

# Collect all pixel values after sqrt stretch
all_pixels = []
num_samples_to_use = 1000  # Use 1000 samples for speed

print(f"\nSampling {num_samples_to_use} images to compute statistics...")

sample_count = 0
for class_name in CLASS_NAMES:
    class_dir = DATA_DIR / class_name
    files = list(class_dir.glob('*.npy'))[:num_samples_to_use//3]
    
    for file_path in tqdm(files, desc=f"Processing {class_name}"):
        image = np.load(file_path)
        if image.shape[0] == 1:
            image = image[0]
        
        # Apply sqrt stretch (same as in training)
        image = np.maximum(image, 0)
        image = np.sqrt(image)
        
        all_pixels.append(image.flatten())
        sample_count += 1

# Concatenate all pixels
all_pixels = np.concatenate(all_pixels)

# Compute statistics
GLOBAL_MEAN = float(all_pixels.mean())
GLOBAL_STD = float(all_pixels.std())

print("\n" + "="*70)
print("COMPUTED STATISTICS")
print("="*70)
print(f"Samples used: {sample_count}")
print(f"Total pixels: {len(all_pixels):,}")
print(f"\n🎯 GLOBAL_MEAN = {GLOBAL_MEAN:.6f}")
print(f"🎯 GLOBAL_STD  = {GLOBAL_STD:.6f}")
print("\nThese are the CORRECT values for your dataset!")
print("="*70)

## Configuration

In [ ]:
class Config:
    # Paths
    DATA_DIR = Path('/content/dataset')
    TRAIN_DIR = DATA_DIR / 'train'
    VAL_DIR = DATA_DIR / 'val'
    CHECKPOINT_DIR = Path('/content/checkpoints')
    LOG_DIR = Path('/content/logs')
    
    # Classes
    CLASS_NAMES = ['no', 'sphere', 'vort']
    CLASS_LABELS = ['No Substructure', 'Subhalo/Sphere', 'Vortex']
    NUM_CLASSES = 3
    
    # Training
    BATCH_SIZE = 32
    NUM_EPOCHS = 100
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.4
    USE_AUGMENTATION = True
    MIN_LR = 1e-7
    EARLY_STOPPING_PATIENCE = 15
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    RANDOM_SEED = 42
    
    # Preprocessing - USE COMPUTED VALUES!
    USE_SQRT_STRETCH = True
    GLOBAL_MEAN = GLOBAL_MEAN  # From computation above
    GLOBAL_STD = GLOBAL_STD    # From computation above
    
    def __init__(self):
        self.CHECKPOINT_DIR.mkdir(exist_ok=True)
        self.LOG_DIR.mkdir(exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.experiment_name = f'resnet34_fixed_v2_{timestamp}'
        self.experiment_dir = self.LOG_DIR / self.experiment_name
        self.experiment_dir.mkdir(exist_ok=True)

config = Config()

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Device: {config.DEVICE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"\n🔧 PREPROCESSING (FIXED):")
print(f"  Mean: {config.GLOBAL_MEAN:.6f}")
print(f"  Std:  {config.GLOBAL_STD:.6f}")
print("="*70)

## Dataset Class

In [ ]:
class GravitationalLensingDataset(Dataset):
    def __init__(self, data_dir, class_names, config, augment=False):
        self.data_dir = Path(data_dir)
        self.class_names = class_names
        self.config = config
        self.augment = augment
        
        self.samples = []
        self.labels = []
        
        print(f"\nLoading data from {data_dir}...")
        for class_idx, class_name in enumerate(class_names):
            class_dir = self.data_dir / class_name
            class_files = sorted(class_dir.glob('*.npy'))
            print(f"  {class_name}: {len(class_files)} samples")
            
            for file_path in class_files:
                self.samples.append(file_path)
                self.labels.append(class_idx)
        
        print(f"Total: {len(self.samples)} samples")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        # Load image
        image = np.load(self.samples[idx])
        if image.shape[0] == 1:
            image = image[0]
        
        # Sqrt stretch
        image = np.maximum(image, 0)
        image = np.sqrt(image)
        
        # FIXED: Global standardization
        image = (image - self.config.GLOBAL_MEAN) / (self.config.GLOBAL_STD + 1e-8)
        
        # To tensor
        image = torch.from_numpy(image).float().unsqueeze(0)
        
        # Augmentation
        if self.augment:
            if np.random.rand() < 0.75:
                k = np.random.randint(0, 4)
                image = torch.rot90(image, k, dims=[-2, -1])
            if np.random.rand() < 0.5:
                image = torch.flip(image, dims=[-1])
            if np.random.rand() < 0.5:
                image = torch.flip(image, dims=[-2])
            if np.random.rand() < 0.3:
                noise = torch.randn_like(image) * 0.02
                image = image + noise
        
        return image, self.labels[idx]

print("✅ Dataset class defined")

## ResNet34 Model

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
    
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)


class ResNet34(nn.Module):
    def __init__(self, num_classes=3, dropout=0.4):
        super().__init__()
        self.in_channels = 64
        
        self.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        
        self.layer1 = self._make_layer(BasicBlock, 64, 3, 1)
        self.layer2 = self._make_layer(BasicBlock, 128, 4, 2)
        self.layer3 = self._make_layer(BasicBlock, 256, 6, 2)
        self.layer4 = self._make_layer(BasicBlock, 512, 3, 2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)
        
        self._init_weights()
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        
        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(block(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

print("✅ ResNet34 model defined")

## Load Data

In [ ]:
torch.manual_seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

train_dataset = GravitationalLensingDataset(
    config.TRAIN_DIR, config.CLASS_NAMES, config, augment=True
)
val_dataset = GravitationalLensingDataset(
    config.VAL_DIR, config.CLASS_NAMES, config, augment=False
)

# FIXED: Set num_workers=0 to avoid multiprocessing errors in Colab
train_loader = DataLoader(
    train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, 
    num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=True
)

print(f"\n✅ Data loaded!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

## Training Setup

In [ ]:
model = ResNet34(config.NUM_CLASSES, config.DROPOUT).to(config.DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=config.MIN_LR)

print("="*70)
print("MODEL SETUP")
print("="*70)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {config.DEVICE}")
print("="*70)

## Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        preds = torch.max(outputs, 1)[1]
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(loader), accuracy_score(all_labels, all_preds)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation', leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            preds = torch.max(outputs, 1)[1]
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(loader), accuracy_score(all_labels, all_preds), all_preds, all_labels

print("✅ Training functions defined")

## Training Loop

In [ ]:
print("="*70)
print("STARTING TRAINING - FIXED v2")
print("="*70)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
patience_counter = 0

for epoch in range(config.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
    print("-" * 50)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"LR: {current_lr:.2e}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, config.DEVICE)
    val_loss, val_acc, _, _ = validate_epoch(model, val_loader, criterion, config.DEVICE)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f"\nSummary:")
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }, config.CHECKPOINT_DIR / 'best_model.pth')
        print(f"  ✅ Best model saved! Acc={val_acc:.4f}")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{config.EARLY_STOPPING_PATIENCE}")
    
    if patience_counter >= config.EARLY_STOPPING_PATIENCE:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best Val Accuracy: {best_val_acc:.4f}")

## Plot Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-', label='Train')
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='Val')
axes[1].axhline(y=best_val_acc, color='g', linestyle='--', label=f'Best: {best_val_acc:.4f}')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.experiment_dir / 'curves.png', dpi=150)
plt.show()

## Evaluate & Save to Drive

In [ ]:
# Load best model
checkpoint = torch.load(config.CHECKPOINT_DIR / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])

# Evaluate
_, val_acc, val_preds, val_labels = validate_epoch(model, val_loader, criterion, config.DEVICE)

# Confusion matrix
cm = confusion_matrix(val_labels, val_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=config.CLASS_LABELS, yticklabels=config.CLASS_LABELS)
plt.title(f'Confusion Matrix (Acc: {val_acc:.4f})')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(config.experiment_dir / 'confusion_matrix.png', dpi=150)
plt.show()

# Classification report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(val_labels, val_preds, target_names=config.CLASS_LABELS, digits=4))

# Save to Drive
import shutil
drive_path = f'/content/drive/MyDrive/lensing_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(drive_path, exist_ok=True)
shutil.copy(config.CHECKPOINT_DIR / 'best_model.pth', drive_path)
shutil.copytree(config.experiment_dir, os.path.join(drive_path, 'logs'))

print(f"\n✅ Results saved to: {drive_path}")